# Session 4 - Reading and Writing Files

this is probably the session most of you will use the most in real work.

90% of bioinformatics is:
1. read a file
2. do something to the data
3. write a new file

today we'll learn how to do steps 1 and 3.

## reading a text file

basic pattern:

```python
with open("filename.txt", "r") as f:
    for line in f:
        print(line)
```

the `with` statement is important - it makes sure the file closes properly even if something goes wrong.

## FASTA files

the most common file format in biology. you've all seen this:

```
>sequence_id description
ATGCGTAGCATGCATGCAT
GCATGCATGCATGC
```

let's write a simple FASTA parser

In [ ]:
# first let's create a test FASTA file to work with
fasta_content = """>gene_001 dnaA chromosomal replication initiator
ATGAGTTTTATTATTAGGTGGCGGTACTGGGAATCGCCTTTACTATCGAA
GCGATTGCCGATATTGGCGATGAAGCGATTGAACGT
>gene_002 gyrA DNA gyrase subunit A
ATGAGCGATTTAGCGATTATCGATGGCGATATCGATGCGATTGATGCGAT
GCGATCGATGCGATGCGATTGCGATGCGATTGCGATC
>gene_003 recA recombinase A
ATGGCTATTCAGATCACTGATCTGGAGATGGGCGAGATTGATGCGATCGA
TGCGATCGATGCGATGCGATTGCGATGCGATTGCG
"""

with open("data/test_sequences.fasta", "w") as f:
    f.write(fasta_content)

print("FASTA file created")

FASTA file created


In [ ]:
# now read and parse it
def parse_fasta(filename):
    sequences = {}
    current_id = None
    current_seq = []
    
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()  # remove whitespace and newlines
            
            if line.startswith(">"):  # header line
                if current_id:  # save previous sequence
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:].split()[0]  # get just the ID
                current_seq = []
            else:
                current_seq.append(line)
        
        # dont forget the last sequence
        if current_id:
            sequences[current_id] = "".join(current_seq)
    
    return sequences

seqs = parse_fasta("data/test_sequences.fasta")

for seq_id, sequence in seqs.items():
    print(f"{seq_id}: {len(sequence)} bp")

gene_001: 87 bp
gene_002: 88 bp
gene_003: 86 bp


## CSV files - the spreadsheet of bioinformatics

CSV = comma separated values. you can open these in Excel too.

example:
```
sample,treatment,od600,viable_count
S1,control,0.42,1.2e8
S2,UV_10min,0.38,8.4e7
```

In [ ]:
import csv

# create a test CSV
data = [
    ["sample", "treatment", "od600", "viable_count"],
    ["S1", "control",   "0.42", "1.2e8"],
    ["S2", "UV_10min",  "0.38", "8.4e7"],
    ["S3", "UV_30min",  "0.21", "3.1e6"],
    ["S4", "UV_60min",  "0.09", "4.2e4"],
    ["S5", "UV_120min", "0.05", "1.1e2"],
]

with open("data/survival_data.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(data)

print("CSV created")

CSV created


In [ ]:
# read the CSV back
with open("data/survival_data.csv", "r") as f:
    reader = csv.DictReader(f)  # DictReader gives you each row as a dict
    
    for row in reader:
        sample = row["sample"]
        treatment = row["treatment"]
        od = float(row["od600"])
        print(f"{sample} ({treatment}): OD600 = {od}")

S1 (control): OD600 = 0.42
S2 (UV_10min): OD600 = 0.38
S3 (UV_30min): OD600 = 0.21
S4 (UV_60min): OD600 = 0.09
S5 (UV_120min): OD600 = 0.05


## writing results to a file

always save your results to a file. don't just print them.

In [ ]:
import csv

# analyse sequences and write results
seqs = parse_fasta("data/test_sequences.fasta")

results = []
for seq_id, seq in seqs.items():
    gc = (seq.count("G") + seq.count("C")) / len(seq) * 100
    results.append({
        "sequence_id": seq_id,
        "length": len(seq),
        "gc_percent": round(gc, 2)
    })

# write to CSV
with open("data/sequence_stats.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["sequence_id", "length", "gc_percent"])
    writer.writeheader()
    writer.writerows(results)

print("results saved to data/sequence_stats.csv")
for r in results:
    print(r)

results saved to data/sequence_stats.csv
{'sequence_id': 'gene_001', 'length': 87, 'gc_percent': 49.43}
{'sequence_id': 'gene_002', 'length': 88, 'gc_percent': 51.14}
{'sequence_id': 'gene_003', 'length': 86, 'gc_percent': 48.84}


## common mistake - file not found

the most common file error. usually means the path is wrong

In [ ]:
with open("sequences.fasta", "r") as f:
    content = f.read()

FileNotFoundError: [Errno 2] No such file or directory: 'sequences.fasta'

In [ ]:
# fix 1: check the path
import os
print(os.getcwd())          # where are you right now?
print(os.listdir("."))      # what files are in current folder?
print(os.listdir("data/"))  # what's in the data folder?

if the file is somewhere else, give the full path:

```python
with open("/home/user/project/data/sequences.fasta", "r") as f:
```

or move to the right directory before running your script

## exercise 4.1

write a function that reads a FASTA file and prints a summary:
- total number of sequences
- total number of bases
- average sequence length
- longest sequence

In [ ]:
def fasta_summary(filename):
    seqs = parse_fasta(filename)
    
    total_seqs = ???
    total_bases = ???
    avg_length = ???
    longest = ???
    
    print(f"sequences : {total_seqs}")
    print(f"total bp  : {total_bases}")
    print(f"avg length: {avg_length:.1f} bp")
    print(f"longest   : {longest} bp")

fasta_summary("data/test_sequences.fasta")